# PyroPredict — Phase 3: Baseline Training (YOLO11 vs YOLO26)

**CMPE 258 Deep Learning** | Spring 2026 | SJSU

This notebook trains two SOTA detectors on the D-Fire wildfire smoke/fire
dataset under identical conditions, evaluates both on the held-out test set,
exports the best weights to ONNX, and saves everything to Google Drive.

**Runtime:** GPU required (T4 or better)  
**Time:** ~60–90 min total (two training runs)

---

### Before running

1. Set runtime to **GPU**: Runtime → Change runtime type → T4 GPU  
2. Have your **Kaggle username + token** ready (same as notebook 01)  
3. Run all cells top to bottom

## 0 — Setup

In [ ]:
!pip install -q ultralytics opencv-python-headless seaborn pyyaml kaggle onnx onnxruntime

import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT = '/content/drive/MyDrive/PyroPredict'
DRIVE_RUNS    = f'{DRIVE_PROJECT}/runs'
DRIVE_EXPORTS = f'{DRIVE_PROJECT}/exports'
DRIVE_METRICS = f'{DRIVE_PROJECT}/metrics'
for d in [DRIVE_RUNS, DRIVE_EXPORTS, DRIVE_METRICS]:
    os.makedirs(d, exist_ok=True)
print(f'Drive project: {DRIVE_PROJECT}')

## 1 — Download D-Fire dataset

In [ ]:
# ┌──────────────────────────────────────────────────────┐
# │  PASTE YOUR KAGGLE USERNAME AND TOKEN BELOW          │
# └──────────────────────────────────────────────────────┘
KAGGLE_USERNAME = "your_username_here"
KAGGLE_TOKEN    = "your_token_here"

import json
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(f'{kaggle_dir}/kaggle.json', 'w') as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_TOKEN}, f)
os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)
print(f'Kaggle credentials saved for: {KAGGLE_USERNAME}')

In [ ]:
%%time
DATA_DIR = '/content/dfire'

if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)
    !kaggle datasets download -d sayedgamal99/smoke-fire-detection-yolo -p /content/
    !unzip -q /content/smoke-fire-detection-yolo.zip -d {DATA_DIR}
    !rm -f /content/smoke-fire-detection-yolo.zip
    print(f'Dataset downloaded to {DATA_DIR}')
else:
    print(f'Dataset already exists at {DATA_DIR}')

In [ ]:
from pathlib import Path
import yaml

data_root = Path(DATA_DIR)

# Auto-detect split directories
def find_split_dirs(root):
    splits = {}
    for split_name in ['train', 'valid', 'val', 'test']:
        for candidate in [root / split_name / 'images', root / 'images' / split_name]:
            if candidate.is_dir() and any(candidate.iterdir()):
                canon = 'val' if split_name == 'valid' else split_name
                splits[canon] = candidate
                break
    if not splits:
        for child in root.iterdir():
            if child.is_dir():
                deeper = find_split_dirs(child)
                if deeper:
                    return deeper
    return splits

split_img_dirs = find_split_dirs(data_root)
print('Detected image dirs:')
for name, p in split_img_dirs.items():
    print(f'  {name}: {p}  ({len(list(p.glob("*")))} files)')

# Write dataset.yaml
dataset_yaml = data_root / 'dataset.yaml'
yaml_cfg = {
    'path': str(data_root.resolve()),
    'train': str(split_img_dirs['train'].relative_to(data_root)),
    'val':   str(split_img_dirs['val'].relative_to(data_root)),
    'test':  str(split_img_dirs['test'].relative_to(data_root)) if 'test' in split_img_dirs else '',
    'nc': 2,
    'names': ['fire', 'smoke'],
}
with open(dataset_yaml, 'w') as f:
    yaml.dump(yaml_cfg, f, default_flow_style=False, sort_keys=False)

print(f'\ndataset.yaml → {dataset_yaml}')
print(dataset_yaml.read_text())

## 2 — Training configuration

Both models are trained with **identical** hyperparameters for a fair comparison.

In [ ]:
# ── Shared training hyperparameters ──────────────────────
TRAIN_CFG = dict(
    data      = str(dataset_yaml),
    epochs    = 50,
    imgsz     = 640,
    batch     = 16,
    patience  = 15,        # early stopping
    optimizer = 'SGD',
    lr0       = 0.01,
    lrf       = 0.01,      # cosine decay to lr0 * lrf
    momentum  = 0.937,
    weight_decay = 0.0005,
    warmup_epochs = 3,
    warmup_momentum = 0.8,
    cos_lr    = True,
    workers   = 2,
    seed      = 42,
    val       = True,
    plots     = True,
    verbose   = True,
)

print('Training config:')
for k, v in TRAIN_CFG.items():
    print(f'  {k:>18s}: {v}')

## 3 — Train YOLO11m baseline

In [ ]:
from ultralytics import YOLO

print('Loading YOLO11m (COCO pre-trained)...')
yolo11 = YOLO('yolo11m.pt')

print('\nStarting YOLO11m training...')
results_11 = yolo11.train(
    **TRAIN_CFG,
    project = '/content/runs',
    name    = 'yolo11m_baseline',
)
print('\nYOLO11m training complete!')

In [ ]:
# Evaluate YOLO11m on test set
print('Evaluating YOLO11m on TEST set...')
yolo11_best = YOLO('/content/runs/yolo11m_baseline/weights/best.pt')
metrics_11 = yolo11_best.val(data=str(dataset_yaml), split='test', plots=True)

print(f'\n{"═"*50}')
print('YOLO11m TEST RESULTS')
print(f'{"═"*50}')
print(f'  mAP@50      : {metrics_11.box.map50:.4f}')
print(f'  mAP@50:95   : {metrics_11.box.map:.4f}')
print(f'  Precision    : {metrics_11.box.mp:.4f}')
print(f'  Recall       : {metrics_11.box.mr:.4f}')
print(f'  F1 (approx)  : {2 * metrics_11.box.mp * metrics_11.box.mr / (metrics_11.box.mp + metrics_11.box.mr + 1e-8):.4f}')

## 4 — Train YOLO26m baseline

In [ ]:
print('Loading YOLO26m (COCO pre-trained)...')
try:
    yolo26 = YOLO('yolo26m.pt')
    YOLO26_AVAILABLE = True
    print('YOLO26m loaded successfully.')
except Exception as e:
    print(f'YOLO26m not available in this ultralytics version: {e}')
    print('Falling back to YOLOv10m as second SOTA model...')
    yolo26 = YOLO('yolov10m.pt')
    YOLO26_AVAILABLE = False

model2_name = 'yolo26m_baseline' if YOLO26_AVAILABLE else 'yolov10m_baseline'

print(f'\nStarting {model2_name} training...')
results_26 = yolo26.train(
    **TRAIN_CFG,
    project = '/content/runs',
    name    = model2_name,
)
print(f'\n{model2_name} training complete!')

In [ ]:
# Evaluate second model on test set
print(f'Evaluating {model2_name} on TEST set...')
yolo26_best = YOLO(f'/content/runs/{model2_name}/weights/best.pt')
metrics_26 = yolo26_best.val(data=str(dataset_yaml), split='test', plots=True)

model2_label = 'YOLO26m' if YOLO26_AVAILABLE else 'YOLOv10m'
print(f'\n{"═"*50}')
print(f'{model2_label} TEST RESULTS')
print(f'{"═"*50}')
print(f'  mAP@50      : {metrics_26.box.map50:.4f}')
print(f'  mAP@50:95   : {metrics_26.box.map:.4f}')
print(f'  Precision    : {metrics_26.box.mp:.4f}')
print(f'  Recall       : {metrics_26.box.mr:.4f}')
print(f'  F1 (approx)  : {2 * metrics_26.box.mp * metrics_26.box.mr / (metrics_26.box.mp + metrics_26.box.mr + 1e-8):.4f}')

## 5 — Side-by-side comparison

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def extract_metrics(metrics, name):
    mp = metrics.box.mp
    mr = metrics.box.mr
    return {
        'Model': name,
        'mAP@50': metrics.box.map50,
        'mAP@50:95': metrics.box.map,
        'Precision': mp,
        'Recall': mr,
        'F1': 2 * mp * mr / (mp + mr + 1e-8),
    }

comparison = pd.DataFrame([
    extract_metrics(metrics_11, 'YOLO11m'),
    extract_metrics(metrics_26, model2_label),
])

print('\n' + '═'*60)
print('BASELINE COMPARISON — TEST SET')
print('═'*60)
print(comparison.to_string(index=False, float_format='{:.4f}'.format))
print('═'*60)

In [ ]:
# Comparison bar chart
metric_cols = ['mAP@50', 'mAP@50:95', 'Precision', 'Recall', 'F1']
x = np.arange(len(metric_cols))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, comparison.iloc[0][metric_cols].values, width,
               label='YOLO11m', color='#3b82f6')
bars2 = ax.bar(x + width/2, comparison.iloc[1][metric_cols].values, width,
               label=model2_label, color='#f59e0b')

ax.set_ylabel('Score')
ax.set_title('Baseline Comparison — YOLO11m vs ' + model2_label)
ax.set_xticks(x)
ax.set_xticklabels(metric_cols)
ax.set_ylim(0, 1.05)
ax.legend()

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.3f}',
                ha='center', va='bottom', fontsize=8)

fig.tight_layout()
fig.savefig(f'{DRIVE_METRICS}/baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6 — Training curves

In [ ]:
# Plot training loss curves side by side
from pathlib import Path

def plot_training_curves(run_dir, model_name, color):
    csv_path = Path(run_dir) / 'results.csv'
    if not csv_path.exists():
        print(f'No results.csv found at {csv_path}')
        return None
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    return df

df_11 = plot_training_curves('/content/runs/yolo11m_baseline', 'YOLO11m', '#3b82f6')
df_26 = plot_training_curves(f'/content/runs/{model2_name}', model2_label, '#f59e0b')

if df_11 is not None and df_26 is not None:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Box loss
    for df, name, color in [(df_11, 'YOLO11m', '#3b82f6'), (df_26, model2_label, '#f59e0b')]:
        box_col = [c for c in df.columns if 'box' in c.lower() and 'train' in c.lower()]
        if box_col:
            axes[0].plot(df['epoch'], df[box_col[0]], label=name, color=color)
    axes[0].set_title('Train Box Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

    # Val mAP@50
    for df, name, color in [(df_11, 'YOLO11m', '#3b82f6'), (df_26, model2_label, '#f59e0b')]:
        map_col = [c for c in df.columns if 'map50' in c.lower() and '95' not in c.lower()]
        if map_col:
            axes[1].plot(df['epoch'], df[map_col[0]], label=name, color=color)
    axes[1].set_title('Val mAP@50')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

    # Val mAP@50:95
    for df, name, color in [(df_11, 'YOLO11m', '#3b82f6'), (df_26, model2_label, '#f59e0b')]:
        map_col = [c for c in df.columns if 'map50-95' in c.lower() or 'map50:95' in c.lower()]
        if map_col:
            axes[2].plot(df['epoch'], df[map_col[0]], label=name, color=color)
    axes[2].set_title('Val mAP@50:95')
    axes[2].set_xlabel('Epoch')
    axes[2].legend()

    fig.tight_layout()
    fig.savefig(f'{DRIVE_METRICS}/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7 — Per-class breakdown

In [ ]:
# Per-class AP for both models
class_names = ['fire', 'smoke']

per_class_rows = []
for model_name, metrics in [('YOLO11m', metrics_11), (model2_label, metrics_26)]:
    ap50_per_class = metrics.box.ap50
    ap_per_class   = metrics.box.ap
    for i, cls in enumerate(class_names):
        if i < len(ap50_per_class):
            per_class_rows.append({
                'Model': model_name,
                'Class': cls,
                'AP@50': ap50_per_class[i],
                'AP@50:95': ap_per_class[i],
            })

df_perclass = pd.DataFrame(per_class_rows)
print('Per-class Average Precision:')
print(df_perclass.to_string(index=False, float_format='{:.4f}'.format))

## 8 — Export to ONNX

In [ ]:
# Export YOLO11m best weights to ONNX
print('Exporting YOLO11m to ONNX...')
yolo11_onnx_path = yolo11_best.export(format='onnx', imgsz=640, simplify=True)
print(f'  → {yolo11_onnx_path}')

print(f'\nExporting {model2_label} to ONNX...')
yolo26_onnx_path = yolo26_best.export(format='onnx', imgsz=640, simplify=True)
print(f'  → {yolo26_onnx_path}')

# Show file sizes
for label, path in [('YOLO11m', yolo11_onnx_path), (model2_label, yolo26_onnx_path)]:
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f'{label} ONNX: {size_mb:.1f} MB')

## 9 — Save everything to Google Drive

In [ ]:
import shutil

# ── Save trained weights (.pt) ──
for run_name in ['yolo11m_baseline', model2_name]:
    src_dir = Path(f'/content/runs/{run_name}')
    dst_dir = Path(f'{DRIVE_RUNS}/{run_name}')
    if src_dir.exists():
        if dst_dir.exists():
            shutil.rmtree(dst_dir)
        shutil.copytree(src_dir, dst_dir)
        print(f'Saved {run_name} → {dst_dir}')

# ── Save ONNX exports ──
for label, path in [('yolo11m_baseline', yolo11_onnx_path),
                     (model2_name, yolo26_onnx_path)]:
    dst = f'{DRIVE_EXPORTS}/{label}.onnx'
    shutil.copy2(path, dst)
    print(f'Saved ONNX → {dst}')

# ── Save comparison metrics ──
comparison.to_csv(f'{DRIVE_METRICS}/baseline_comparison.csv', index=False)
df_perclass.to_csv(f'{DRIVE_METRICS}/perclass_ap.csv', index=False)
print(f'\nMetrics saved to {DRIVE_METRICS}')

In [ ]:
# Final summary
print(f'\n{"═"*60}')
print('PHASE 3 COMPLETE — BASELINE TRAINING')
print(f'{"═"*60}')
print(f'\nAll artifacts saved to Google Drive: {DRIVE_PROJECT}')
print(f'\nDirectory contents:')
for root, dirs, fls in os.walk(DRIVE_PROJECT):
    level = root.replace(DRIVE_PROJECT, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 3:
        for f in sorted(fls):
            size = os.path.getsize(os.path.join(root, f)) / (1024*1024)
            print(f'{indent}  {f} ({size:.1f} MB)')

print(f'\n{"═"*60}')
print('RESULTS SUMMARY')
print(f'{"═"*60}')
print(comparison.to_string(index=False, float_format='{:.4f}'.format))
print(f'{"═"*60}')
print(f'\nNext → Run 03_ablations.ipynb to improve these baselines.')